In [ ]:
# !pip install pyspark
from pyspark.sql import SparkSession
ss = SparkSession.builder \
    .master("local[*]") \
    .appName("classification") \
    .getOrCreate()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path_train = "/content/drive/MyDrive/DDAM project/data/train_test/train_set.parquet"
path_test  = "/content/drive/MyDrive/DDAM project/data/train_test/test_set.parquet"

df_train = ss.read.parquet(path_train)
df_test = ss.read.parquet(path_test)

In [ ]:
df_train.show()

## Drop columns

In [ ]:
columns_to_drop = ["XCoords", "YCoords", "Image", "Tile", "Date"]

df_train = df_train.drop(*columns_to_drop)
df_test = df_test.drop(*columns_to_drop)

In [ ]:
df_train.show()

## Confidence Conversion

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# Converts Confidence (string) to a numeric index
confidence_indexer = StringIndexer(
    inputCol="Confidence",
    outputCol="Confidence_indexed",
    handleInvalid="keep"
)

# One-Hot Encoding of the Confidence index
confidence_ohe = OneHotEncoder(
    inputCol="Confidence_indexed",
    outputCol="Confidence_ohe",
    dropLast=False  # Retains all categories
)

## Random search Cross Validation



In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline
from xgboost.spark import SparkXGBClassifier

feature_cols = [
    "NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI", "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"
]

label = "Class"

# Indexes the target column into numerical labels
indexer = StringIndexer(inputCol=label, outputCol="label")

# Assembles the features into a single column vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Defines the XGBoost classifier
xgb_estimator = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    num_workers=2,
    seed=42
)

In [ ]:
import random
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

pipeline = Pipeline(stages=[confidence_indexer, confidence_ohe, indexer, assembler, xgb_estimator])

# Defines a grid larger than NUM_SAMPLES to enable random sampling
large_param_grid = (
    ParamGridBuilder()
    .addGrid(xgb_estimator.max_depth, [4, 6, 8])        # 3 options
    .addGrid(xgb_estimator.subsample, [0.7, 0.8, 1.0])  # 3 options
    .addGrid(xgb_estimator.n_estimators, [50, 100])     # 2 options
    .build()  # Total: 3 * 3 * 2 = 18 combinations
)

NUM_SAMPLES = 8
SEED = 42

# Randomly samples NUM_SAMPLES combinations from the full grid
sampled_paramGrid = random.sample(large_param_grid, NUM_SAMPLES)

crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=sampled_paramGrid,
    evaluator=evaluator,
    numFolds=5,
    seed=SEED
)

print(f"\n Avvio Random Search Cross-Validation ({NUM_SAMPLES} combinazioni)...")

cvModel = crossval.fit(df_train)

# Extract the best model and the average CV metrics
bestModel = cvModel.bestModel
avgMetrics = cvModel.avgMetrics

print(f"\n Cross-Validation completata. Best F1: {max(avgMetrics):.4f}")

In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/DDAM project/models/best_xgb_model"

# Save on Drive
print(f"Salvataggio del modello in corso su: {OUTPUT_PATH} ...")
bestModel.write().overwrite().save(OUTPUT_PATH)

print("Modello salvato con successo!")


In [ ]:
from pyspark.sql.functions import col

# Apply the best model to the test set
predictions = bestModel.transform(df_test)
predictions.cache()
predictions.count()  # Forces materialization (cache is lazy)

# Preview of the predictions
predictions.select(col("label"), col("prediction")).show(5)

metrics_to_evaluate = ["accuracy", "f1", "weightedPrecision", "weightedRecall"]

print("\n--- Metriche di Performance (Test Set) ---")

for metric_name in metrics_to_evaluate:
    test_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric_name
    )
    metric_value = test_evaluator.evaluate(predictions)
    print(f"  {metric_name.ljust(20)}: {metric_value:.4f}")

# --- Confusion Matrix ---
print("\nConfusion Matrix")

confusion_matrix = predictions.groupBy('label').pivot('prediction').count().fillna(0).orderBy('label')
confusion_matrix.show()


In [ ]:
from pyspark.ml.feature import IndexToString
from pyspark.mllib.evaluation import MulticlassMetrics


# Converts the numerical prediction indices back to the original string labels
labelConverter = IndexToString(
    inputCol="prediction",
    outputCol="predictedLabel",
    labels=bestModel.stages[2].labels
)


predictions_with_strings = labelConverter.transform(predictions)
predictions_with_strings.cache()
predictions_with_strings.count()


# Actual count per class
support_counts_df = predictions_with_strings.groupBy("label").count()
support_map = support_counts_df.rdd.collectAsMap()


# MulticlassMetrics requires an RDD of (prediction, label) pairs as Double
prediction_and_labels = (
    predictions_with_strings.select(col("prediction"), col("label"))
    .rdd.map(lambda row: tuple(map(float, row)))
)


metrics = MulticlassMetrics(prediction_and_labels)
labels = predictions_with_strings.select("label").distinct().rdd.flatMap(lambda x: x).collect()


print(f"{'Classe':<25} {'Support':<10} {'Precision':<15} {'Recall':<15} {'F1-Score':<15}")
print("-" * 80)


for label_index in sorted(labels):
    class_precision = metrics.precision(label=label_index)
    class_recall    = metrics.recall(label=label_index)
    class_f1        = metrics.fMeasure(label=label_index)
    class_support   = support_map.get(label_index, 0)
    class_string    = bestModel.stages[2].labels[int(label_index)]


    print(f"{class_string:<25.25} {class_support:<10} {class_precision:<15.4f} {class_recall:<15.4f} {class_f1:<15.4f}")


print("-" * 80)


predictions_with_strings.unpersist()
predictions.unpersist()


# **Variabile Target Binaria**

In [ ]:
# !pip install pyspark
from pyspark.sql import SparkSession
ss = SparkSession.builder \
    .master("local[*]") \
    .appName("classification") \
    .getOrCreate()


In [ ]:
path_train = "/content/drive/MyDrive/DDAM project/data/train_test/train_set.parquet"
path_test  = "/content/drive/MyDrive/DDAM project/data/train_test/test_set.parquet"

df_train = ss.read.parquet(path_train)
df_test = ss.read.parquet(path_test)

In [ ]:
df_train.show()

In [ ]:
from pyspark.sql import functions as F

# Marine Debris = 1, Others = 0
df_train = df_train.withColumn(
  'Class',
  F.when(F.col('Class') == 'Marine Debris', 1).otherwise(0)
)

df_test = df_test.withColumn(
  'Class',
  F.when(F.col('Class') == 'Marine Debris', 1).otherwise(0)
)

## Drop columns

In [ ]:
columns_to_drop = ["XCoords", "YCoords", "Image", "Tile", "Date"]

df_train = df_train.drop(*columns_to_drop)
df_test = df_test.drop(*columns_to_drop)

In [ ]:
df_train.show()

## Confidence Conversion

In [ ]:
# Converts Confidence (string) to a numeric index
confidence_indexer = StringIndexer(
    inputCol="Confidence",
    outputCol="Confidence_indexed",
    handleInvalid="keep"
)

# One-Hot Encoding of the Confidence index
confidence_ohe = OneHotEncoder(
    inputCol="Confidence_indexed",
    outputCol="Confidence_ohe",
    dropLast=False  # Retains all categories
)

## Random search Cross Validation



In [ ]:
feature_cols = [
    "NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI", "CON", "DIS", "ENER", "COR", "PC_1", "Confidence_ohe"
]


label = "Class"


# Indexes the target column into numerical labels
indexer = StringIndexer(inputCol=label, outputCol="label")


# Assembles the features into a single column vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")


# Defines the XGBoost classifier
xgb_estimator = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    num_workers=2,
    seed=42
)


In [ ]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)


pipeline = Pipeline(stages=[confidence_indexer, confidence_ohe, indexer, assembler, xgb_estimator])


# Defines a grid larger than NUM_SAMPLES to enable random sampling
large_param_grid = (
    ParamGridBuilder()
    .addGrid(xgb_estimator.max_depth, [4, 6, 8])
    .addGrid(xgb_estimator.subsample, [0.7, 0.8, 1.0])
    .addGrid(xgb_estimator.n_estimators, [50, 100])
    .build()  # Total: 3 * 3 * 2 = 18 combinations
)


NUM_SAMPLES = 8
SEED = 42


# Randomly samples NUM_SAMPLES combinations from the full grid
sampled_paramGrid = random.sample(large_param_grid, NUM_SAMPLES)


crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=sampled_paramGrid,
    evaluator=evaluator,
    numFolds=5,
    seed=SEED
)


print(f"\n Starting Random Search Cross-Validation ({NUM_SAMPLES} combinations))


cvModel = crossval.fit(df_train)


# Extract the best model and the average CV metrics
bestModel = cvModel.bestModel
avgMetrics = cvModel.avgMetrics


print(f"\n Cross-Validation completed. Best F1: {max(avgMetrics):.4f}")


In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/DDAM project/models/best_xgb_binary_model"

# Save on Drive
print(f"Salvataggio del modello in corso su: {OUTPUT_PATH} ...")
bestModel.write().overwrite().save(OUTPUT_PATH)

print("Model saved")


In [ ]:
# Apply the best model to the test set
predictions = bestModel.transform(df_test)
predictions.cache()
predictions.count()  # Forces materialization (cache is lazy)


# Preview of the predictions
predictions.select(col("label"), col("prediction")).show(5)


metrics_to_evaluate = ["accuracy", "f1", "weightedPrecision", "weightedRecall"]


print("\n--- Performance Metrics (Test Set) ---")


for metric_name in metrics_to_evaluate:
    test_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric_name
    )
    metric_value = test_evaluator.evaluate(predictions)
    print(f"  {metric_name.ljust(20)}: {metric_value:.4f}")


# --- Confusion Matrix ---
print("\n--- Confusion Matrix ---")


confusion_matrix = predictions.groupBy('label').pivot('prediction').count().fillna(0).orderBy('label')
confusion_matrix.show()


In [ ]:
from pyspark.ml.feature import IndexToString
from pyspark.mllib.evaluation import MulticlassMetrics


# Converts the numerical prediction indices back to the original string labels
labelConverter = IndexToString(
    inputCol="prediction",
    outputCol="predictedLabel",
    labels=bestModel.stages[2].labels
)


predictions_with_strings = labelConverter.transform(predictions)
predictions_with_strings.cache()
predictions_with_strings.count()


# Actual count per class -> dictionary for fast lookup in the loop
support_counts_df = predictions_with_strings.groupBy("label").count()
support_map = support_counts_df.rdd.collectAsMap()


# MulticlassMetrics requires an RDD of (prediction, label) pairs as Double
prediction_and_labels = (
    predictions_with_strings.select(col("prediction"), col("label"))
    .rdd.map(lambda row: tuple(map(float, row)))
)


metrics = MulticlassMetrics(prediction_and_labels)
labels = predictions_with_strings.select("label").distinct().rdd.flatMap(lambda x: x).collect()


print("\n--- Per-Class Metrics (Test Set) ---")
print(f"{'Class':<25} {'Support':<10} {'Precision':<15} {'Recall':<15} {'F1-Score':<15}")
print("-" * 80)


for label_index in sorted(labels):
    class_precision = metrics.precision(label=label_index)
    class_recall    = metrics.recall(label=label_index)
    class_f1        = metrics.fMeasure(label=label_index)
    class_support   = support_map.get(label_index, 0)
    class_string    = bestModel.stages[2].labels[int(label_index)]


    print(f"{class_string:<25.25} {class_support:<10} {class_precision:<15.4f} {class_recall:<15.4f} {class_f1:<15.4f}")


print("-" * 80)


predictions_with_strings.unpersist()
predictions.unpersist()
